In [ ]:
from google import genai
from google.genai import types
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.embeddings import Embeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.stores import InMemoryStore
from langchain_core.documents import Document
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_google_genai  import ChatGoogleGenerativeAI
from langchain_community.retrievers import BM25Retriever
from sentence_transformers import CrossEncoder

import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

C:\Users\cmanw\AppData\Local\Temp\ipykernel_17908\808463896.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


In [ ]:
# initialize client once (reuse across calls)
# client = genai.Client(api_key=api_key)

# def embed_text(context=None, model="gemini-embedding-2"):
#     """
#     Embed a single text string. If `context` provided, it's prepended to `text`.
#     Returns: list[float] embedding vector.
#     """
   
#     resp = client.models.embed_content(model=model, contents=context)
#     return resp.embeddings[0].values

# # example
# vec = embed_text("This doc is about animals")
# print(len(vec), vec[:10])

In [ ]:
from pathlib import Path

pdf_dir = Path(r"C:\Users\cmanw\OneDrive\Documents\AI-Projects\Project_1\PDF_FOLDER")
if not pdf_dir.exists():
    raise FileNotFoundError(f"PDF folder not found: {pdf_dir}")

# Use the directory loader when pointing to a folder of PDFs
loader = PyPDFDirectoryLoader(str(pdf_dir))
docs = loader.load()

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=30)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)


child_docs = child_splitter.split_documents(docs)
parent_docs = parent_splitter.split_documents(docs)
# child_docs[:5]


In [3]:

print(len(child_docs))

texts = child_docs


81


In [4]:
# docstore = InMemoryStore()

class GeminiEmbeddings(Embeddings):
    def __init__(self, api_key, model="gemini-embedding-2"):
        self.client = genai.Client(api_key=api_key)
        self.model = model

    def embed_documents(self, texts):
        embeddings = []
        for text in texts:
            resp = self.client.models.embed_content(model=self.model, contents=[text])
            embeddings.append(resp.embeddings[0].values)    
        return embeddings

    def embed_query(self, text):
        resp = self.client.models.embed_content(model=self.model, contents=[text])
        return resp.embeddings[0].values

gemini_embeddings = GeminiEmbeddings(api_key=api_key)


# retriever = ParentDocumentRetriever(
#     vectorstore=vectorstore,
#     parent_splitter=parent_splitter,
#     child_splitter=child_splitter,
#     docstore=docstore,
# )
# retriever.add_documents(docs)

# query = "What is LangChain?"
# results = retriever.invoke(query)

In [ ]:
embeddings = gemini_embeddings.embed_documents(texts)
# embeddings[0]
print(len(texts))
print(len(embeddings))

In [ ]:
docstore = InMemoryStore()
vectorstore = Chroma(
    embedding_function=gemini_embeddings,
    persist_directory="./chroma_db"
)

In [ ]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    parent_splitter=parent_splitter,
    child_splitter=child_splitter, 
    docstore=docstore
)

# Increase child retrieval
retriever.search_kwargs = {"k": 20}

# BM25
bm25_retriever = BM25Retriever.from_documents(docstore)
bm25_retriever.k = 5

query = "What's AI agent?"

parent_results = retriever.invoke(query)
bm25_results = bm25_retriever.invoke(query)



[Document(metadata={'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationdate': '2025-04-07T14:20:51+00:00', 'moddate': '2025-04-07T14:20:54+00:00', 'source': 'C:\\Users\\cmanw\\OneDrive\\Documents\\AI-Projects\\Project_1\\PDF_FOLDER\\a-practical-guide-to-building-agents.pdf', 'total_pages': 34, 'page': 3, 'page_label': '4'}, page_content="What is an \nagent?\nWhile conventional software enables users to streamline and automate workflows, agents are able \nto perform the same workflows on the users’ behalf with a high degree of independence.\nAgents are systems that independently accomplish tasks on your behalf.\nA workflow is a sequence of steps that must be executed to meet the user’s goal, whether that's \nresolving a customer service issue, booking a restaurant reservation, committing a code change, \u2028\nor generating a report.\nApplications that integrate LLMs but don’t use them to control workflow execu

In [ ]:
encoder_model = CrossEncoder("BAAI/bge-reranker-base")
all_docs = parent_results + bm25_results

unique_docs = []

seen = set()

for doc in all_docs:

    if doc.page_content not in seen:
        unique_docs.append(doc)
        seen.add(doc.page_content)

"What is an \nagent?\nWhile conventional software enables users to streamline and automate workflows, agents are able \nto perform the same workflows on the users’ behalf with a high degree of independence.\nAgents are systems that independently accomplish tasks on your behalf.\nA workflow is a sequence of steps that must be executed to meet the user’s goal, whether that's \nresolving a customer service issue, booking a restaurant reservation, committing a code change, \u2028\nor generating a report.\nApplications that integrate LLMs but don’t use them to control workflow execution—think simple \nchatbots, single-turn LLMs, or sentiment classifiers—are not agents.\nMore concretely, an agent possesses core characteristics that allow it to act reliably and \nconsistently on behalf of a user:\n01 It leverages an LLM to manage workflow execution and make decisions. It recognizes \nwhen a workflow is complete and can proactively correct its actions if needed. In case \u2028\nof failure, it 

In [ ]:
pairs = [
    (query, doc.page_content)
    for doc in unique_docs
]
scores = encoder_model.predict(pairs)

doc_scores = list(zip(all_docs, scores))

doc_scores.sort(
    key=lambda x: x[1],
    reverse=True
)

final_docs = [
    doc
    for doc, score in doc_scores[:5]
]

context = "\n\n".join(
    doc.page_content
    for doc in final_docs
)

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    api_key=api_key,
)

response = llm.invoke(
    f"""
Context:
{context}

Question:
{query}
"""
)

In [23]:
response

AIMessage(content=[{'type': 'text', 'text': "Based on the context provided, an **AI agent** is a system that performs tasks and executes workflows on a user’s behalf with a high degree of independence.\n\nUnlike conventional software, which merely helps a user streamline a process, or simple LLM applications (like basic chatbots) that do not control workflow execution, an agent is characterized by two core traits:\n\n1.  **LLM-Driven Decision Making:** It uses an LLM to manage the sequence of steps required to reach a goal. It can recognize when a task is finished, proactively correct its own mistakes, and halt execution to ask for human help if it fails.\n2.  **Tool Use:** It has access to external tools to gather information or perform actions. It can dynamically select the right tool for the current situation while operating within specific safety guardrails.\n\nIn short, an agent is an autonomous system that doesn't just assist with a workflow—it actively executes the workflow to a